In [1]:
# baixa o arquivo de texto do shakespeare
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-05-20 15:39:05--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-05-20 15:39:06 (23.0 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
with open('input.txt', 'r', encoding='utf-8') as f: # abre o arquivo para leitura
    text = f.read() # lê todo o conteúdo do arquivo

In [3]:
print("length of dataset in characters: ", len(text)) # exibe o total de caracteres

length of dataset in characters:  1115394


In [5]:
print(text[:1000]) # mostra o texto

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [6]:
chars = sorted(list(set(text))) # cria uma lista ordenada
vocab_size = len(chars) # guarda a quantidade total de caracteres
print("".join(chars)) # exibe todos os caracteres únicos
print(vocab_size) # mostra na tela o tamanho total


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [7]:
stoi = { ch:i for i,ch in enumerate(chars)} # cria um dicionário que mapeia cada caractere
itos = { i:ch for i,ch in enumerate(chars)} # cria um dicionário que mapeia cada número inteiro
encode = lambda s: [stoi[c] for c in s] # define a função que converte uma string de texto
decode = lambda l: "".join([itos[i] for i in l]) # define a função que converte uma lista
print(encode("hii there")) # converte a frase em números e exibe
print(decode(encode("hii there"))) # codifica a frase em números

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [9]:
import torch # importa a biblioteca pytorch para trabalhar com tensores
data = torch.tensor(encode(text), dtype=torch.long) # converte todo o texto codificado em numeros
print(data.shape, data.dtype) # exibe o tamanho total do tensor
print(data[:1000])# exibe apenas os primeiros 1000 números do tensor criado

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [10]:
n = int (0.9*len(data)) # pega o noventea porcento
train_data = data[:n] # utiliza estes 90% para treinammento
val_data = data[n:] # utiliza o resto

In [11]:
block_size = 8 # cada bloco de dados tera 8 elementos
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [13]:
x = train_data[:block_size] #seleciona os 8 primeiros elementos
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1] #percorre o bloco de codigo ate o seu ultimo
    target = y[t] # proximo caractere a ser previsto
    print(f"when input {context} the target {target}")

when input tensor([18]) the target 47
when input tensor([18, 47]) the target 56
when input tensor([18, 47, 56]) the target 57
when input tensor([18, 47, 56, 57]) the target 58
when input tensor([18, 47, 56, 57, 58]) the target 1
when input tensor([18, 47, 56, 57, 58,  1]) the target 15
when input tensor([18, 47, 56, 57, 58,  1, 15]) the target 47
when input tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target 58


In [14]:
torch.manual_seed(1337) # define a semente aleatoria
batch_size = 4 # define quantas sequências serão processadas em paralelo por vez
block_size = 8 # define o tamanho máximo do contexto

def get_batch(split): # define a função que cria lotes
    data = train_data if split == 'train' else val_data # seleciona o conjunto de dados
    ix = torch.randint(len(data) - block_size, (batch_size,)) # sorteia índices iniciais aleatorios
    x = torch.stack([data[i:i+block_size] for i in ix]) # recorta os blocos de texto
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) # recorta os mesmos blocos deslocados
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # percorre cada uma das sequencias
    for t in range(block_size): # percorre cada caractere ao longo do tempo
        context = xb[b, :t+1] # extrai todo o histórico de caracteres até a posicao
        target = yb[b, t] # obtém o caractere que vem logo em seguida
        print(f"when input is {context.tolist()} the target: {target}") # exibe a relação entre o contexto

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [15]:
print(xb) #imprime o transformador

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [19]:
import torch # importa a biblioteca principal do pytorch
import torch.nn as nn # importa o módulo de redes neurais
from torch.nn import functional as F # importa funções utilitárias como ativações e perdas
torch.manual_seed(1337) # define a semente aleatori

class BigramLanguageModel(nn.Module): # define a classe do modelo de linguagem

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)# cria uma tabela com os embedding

    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx) # gera tensor
        if targets is None: # caso nao tenha palavras corretas
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    # e o maximo de tokens que o modelo deve gerar
    # pega a ultima palavra do contexto para prever a proxima
    def generate(self, idx, max_new_tokens):
            for _ in range(max_new_tokens):
                logits, loss = self(idx) # faz uma previsao baseada no texto atua
                logits = logits[:, -1, :] # becomes (B, C)
                probs = F.softmax(logits, dim=-1) # (B, C)
                idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
                idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
            return idx # retorna a sequencia

m = BigramLanguageModel(vocab_size) # cria uma instância do modelo
out = m(xb, yb) # passa os lotes de entrada
print(out[0].shape) # exibe o formato das dimensoes

# comeca com o texto no indice 0
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [20]:
# criando um otimizacor para o modelo
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [22]:
batch_size = 32 # define o tamanho do lote
for steps in range(10000): # executa o laço de treinamento por 10000 iteracoes

    xb, yb = get_batch('train') # sorteia um novo lote de dados de treino

    # calcula os logits e a funcao de perda
    logits, loss = m(xb, yb) # passa os dados pelo modelo para calcular
    optimizer.zero_grad(set_to_none=True) # limpa a memória dos gradientes
    loss.backward() # calcula o impacto de cada peso
    optimizer.step() # atualiza os pesos do modelo para reduzir o erro na proximp

print(loss.item())

2.415212392807007


In [25]:
# converte para uma lista, e o encode transforma em frase
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


TETh'dofou me ylo ai's Th ble ut ind Ise, are abel t wivive ou
I t fan the yowamarealoray en my aile d y tl s Vissthowooweing, brorin s,

I phestome, hengerin no ng-s, melalif t-tyo t io ee s athepe, live
Yond wes mu s l rondind vit; t heroulfe ck,

'gs fe S:

Selama me atllecer, thincof meed che t d Prcheeay TINGSIn housu I'ssin
Freshavend gedilsu bonshainore it f baven stlt's.
Exchoote ghet.
NE: o ely heree hat han;
Ke p----po'To INar is?

Wo bollare athajo t:
KESe ner
KI bul we wod n:
ATot, t


## The mathematical trick in self-attention

In [24]:
torch.manual_seed(1337) # fixa a semente aleatória para garantir
B,T,C = 4,8,2 # define as dimensões para o lote
x = torch.randn(B,T,C) # cria um tensor preenchido com números aleatorios
x.shape # retorna o formato das dimensões do tensor

torch.Size([4, 8, 2])

In [28]:
xbow = torch.zeros((B,T,C)) # cria um tensor de zeros com o mesmo tamanho
for b in range(B): # percorre cada exemplo do lote
    for t in range(T): # percorre cada etapa de tempo e caractere na sequencia
        xprev = x[b,:t+1] # # (t,C) # extrai todos os elementos da sequenci
        xbow[b,t] = torch.mean(xprev, 0) # calcula a media desses elementos

In [39]:
wei = torch.tril(torch.ones(T, T)) # cria uma matriz triangular inferior preenchida
wei = wei / wei.sum(1, keepdim=True) # normaliza cada linha dividindo pela soma
wbow2 = wei @ x
torch.allclose(wbow2, xbow)

False

In [59]:
torch.manual_seed(42) # fixa a semente aleatória para garantir que os numeros
a = torch.ones(3, 3) # cria uma matriz de tamanho 3x3
b = torch.randint(0,10,(3,2)).float() # cria uma matriz 3x2 com números inteiro
c = a @ b # realiza a multiplicação matricial
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[14., 16.],
        [14., 16.],
        [14., 16.]])


In [77]:
tril = torch.tril(torch.ones(T, T)) # cria uma matriz triangular inferior
wei = torch.zeros((T,T)) # cria uma matriz de pesos inicializada
wei = wei.masked_fill(tril == 0, float('-inf')) # substitui os valores acima da diagonal
wei = F.softmax(wei, dim=-1) # aplica a função softmax para transformar as pontuacoes
xbow3 = wei @ x # realiza a média ponderada do histórico multiplicando a matriz

# Recalculate xbow with the current B, T, C and x to ensure correct comparison
_xbow_recalculated = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        _xbow_recalculated[b,t] = torch.mean(xprev, 0)

torch.allclose(_xbow_recalculated, xbow3)

False

In [60]:
torch.manual_seed(1337) # fixa a semente aleatória para consistência
B,T,C = 4,8,32 # define o lote em 4, o tempo em 8 e o número de canais aumentado para 3
x = torch.randn(B,T,C) # cria o tensor de entrada aleatorio

head_size = 16 # define o tamanho da projeção interna
key = nn.Linear(C, head_size, bias=False) # cria a camada linear para gerar as chaves
query = nn.Linear(C, head_size, bias=False) # cria a camada linear para gerar as perguntas
value = nn.Linear(C, head_size, bias=False) # cria a camada linear para gerar os valores
k = key(x) # projeta as entradas x para gerar as chaves k
q = query(x) # projeta as entradas x para gerar as consultas q
wei = q @ k.transpose(-2, -1)  # calcula a afinidade multiplicando as consulta

tril = torch.tril(torch.ones(T, T)) # recria a matriz de mascara
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf')) # aplica a mascara preenchendo
wei = F.softmax(wei, dim=-1) # normaliza as pontuações de afinidade

v = value(x) # projeta as entradas para extrair os valores reais
out = wei @ v # calcula a saída ponderando os valores pelas probabilidades
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [72]:
wbow2[0], xbow[0]

(tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

In [76]:
tril = torch.tril(torch.ones(T, T)) # cria uma matriz de máscara isolando
wei = torch.zeros((T,T)) # inicializa a matriz
wei = wei.masked_fill(tril == 0, float('-inf')) # impede que o modelo olhe para o futuro
wei = F.softmax(wei, dim=-1) # converte as pontuações em probabilidades
xbow3 = wei @ x # realiza a agregação dos dados multiplicando os pesos

# Recalculate xbow with the current B, T, C and x to ensure correct comparison
_xbow_recalculated = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        _xbow_recalculated[b,t] = torch.mean(xprev, 0)

torch.allclose(_xbow_recalculated, xbow3, atol=1e-4)

True

In [66]:
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [65]:
wei

tensor([[[  7.9429,  -6.9481,   1.6754,   1.1938,  -2.1804,   1.9768,  -2.9069,
           -1.9238],
         [ -2.4603,   3.0816,   0.4861,   0.4772,  -4.2238,  -0.4934,   1.5671,
           -1.0749],
         [  1.8043,   2.6402,   3.4943,  -2.0260,   3.4379,   0.9933,   2.8380,
           -1.2965],
         [  5.7401,  -2.2395,   4.8651,  -0.3254,   6.9252,   1.3686,  -1.2585,
           -3.6711],
         [ -8.0815,   7.4866,  -4.4856,  -0.5266,  -1.7281,   3.3845,   4.3966,
            7.4605],
         [  4.0002,   2.1576,   3.9230,  -0.3601,   2.9457,   5.2070,   5.9117,
            4.9540],
         [  4.2167,  -2.0995,   0.5030,  -0.3124,   3.2943,  -4.2182,   1.4404,
           -2.2715],
         [  1.0349,   0.6481,   2.5885,   1.1346,   5.0565,   1.5558,  -2.4873,
           -1.8402]],

        [[  1.1832,   1.5408,  -2.9824,   0.1945,  -0.6890,  -1.2215,   3.3195,
            1.7454],
         [ -3.4201,   1.0540,   4.3044,   3.8175,   3.0116,  -3.8022,   1.0848,
         

Notes:

- Attention is a communication mechanism. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.  

- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.  

- Each example across batch dimension is of course processed completely independently and never "talk" to each other

- In an "encoder" attention block just delete the single line that does masking with tril, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling."/

- "self-attention" just means that the keys and values are produced from the same source as queries.In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)

- "Scaled" attention additional divides wei by $1/\text{sqrt}(\text{head\_size})$. This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [61]:
k = torch.randn(B,T,head_size) # gera o tensor de chaves com valores
q = torch.randn(B,T,head_size) # gera o tensor de consultas com valores
wei = q @ k.transpose(-2, -1) # multiplica q por k transposto para calcular a afinidade

In [62]:
k.var() # calcula e exibe a variância do tensor k

tensor(1.0449)

In [63]:
q.var() # calcula e exibe a variância do tensor q

tensor(1.0700)

In [64]:
wei.var() # mostra que a variância de wei

tensor(17.4690)

In [78]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1) # calcula o softmax com valores pequenos

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [79]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # multiplica por 8 antes do softmax

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [81]:
class LayerNorm1d:

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps # salvando o valor
    self.gamma = torch.ones(dim) # tensor gamma com o tamanho
    self.beta = torch.zeros(dim) # tensor beta com o tamanho

  def __call__(self, x): # onde a normalizacao ocorrera
    xmean = x.mean(1, keepdim=True) # calcula a media de cada linha dos dados
    xvar = x.var(1, keepdim=True) # caclula a variancia de cada linha dos dados
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normaliza os dados
    self.out = self.gamma * xhat + self.beta # aplicando os ajustes de escala
    return self.out #valor normlizado

  def parameters(self): # usado para obter os parametros da camada
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100) # 100 caracteristeicas para normalizar
x = torch.randn(32, 100) # 32 exemplos e cada exemplo tem 100 caracteristicas
x = module(x) # passando os dados de x para module
x.shape

torch.Size([32, 100])

In [82]:
x[:,0].mean(), x[:,0].std() # calcula a media e o desvio padrao

(tensor(0.1469), tensor(0.8803))

In [83]:
x[0,:].mean(), x[0,:].std() #calcula a media e o desvio padrao

(tensor(-9.5367e-09), tensor(1.0000))